# Leakage Examples in Validation

This notebook demonstrates three common leakage patterns with **fully synthetic**, seed-controlled data:

1. **Target leakage** from full-data target encoding/statistics
2. **Temporal leakage** from random split on time-dependent data
3. **Group leakage** from entity overlap between train and test

For each case, we build a **flawed pipeline**, a **corrected pipeline**, compare metrics, and interpret the result.

## Reproducibility and dependencies

- Randomness is controlled via a fixed `SEED`.
- Only common libraries are used: `numpy`, `pandas`, `sklearn`, `matplotlib`.
- All examples are self-contained and runnable end-to-end.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.style.use('seaborn-v0_8-whitegrid')

## 1) Target leakage via full-data encoding/statistics

### Setup
We create a binary classification dataset with a **row-level ID** (`record_id`) and a random target.

- `record_id` has no real predictive value.
- If we compute target-mean encoding using the **entire dataset before splitting**, each row's encoded value includes its own target label.
- This leaks answer information into the feature matrix.

In [ ]:
n_rows = 4000

df_target = pd.DataFrame({
    'record_id': [f'id_{i}' for i in range(n_rows)],
    'noise_feature': rng.normal(0, 1, n_rows),
})
df_target['target'] = rng.binomial(1, 0.5, n_rows)

print(df_target.head())
print(f"Target rate: {df_target['target'].mean():.3f}")

### Flawed pipeline (leaks target)

`record_id` target-mean is computed on full data **before** splitting.

In [ ]:
# Leakage: using target values from all rows (including future test rows)
full_target_means = df_target.groupby('record_id')['target'].mean()
df_target['id_te_full_data'] = df_target['record_id'].map(full_target_means)

X_flawed = df_target[['id_te_full_data', 'noise_feature']]
y_flawed = df_target['target']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_flawed, y_flawed, test_size=0.3, random_state=SEED, stratify=y_flawed
)

model_target_flawed = LogisticRegression(solver='liblinear', random_state=SEED)
model_target_flawed.fit(X_train_f, y_train_f)

pred_prob_f = model_target_flawed.predict_proba(X_test_f)[:, 1]
pred_cls_f = (pred_prob_f >= 0.5).astype(int)

aUC_target_flawed = roc_auc_score(y_test_f, pred_prob_f)
acc_target_flawed = accuracy_score(y_test_f, pred_cls_f)

print(f"Flawed ROC-AUC: {aUC_target_flawed:.4f}")
print(f"Flawed Accuracy: {acc_target_flawed:.4f}")

### Corrected pipeline (train-only encoding)

We split first, compute target-mean encoding **only on training data**, and map test rows with a train-derived fallback (global mean).

In [ ]:
idx_train, idx_test = train_test_split(
    df_target.index,
    test_size=0.3,
    random_state=SEED,
    stratify=df_target['target']
)

train_target = df_target.loc[idx_train].copy()
test_target = df_target.loc[idx_test].copy()

train_means = train_target.groupby('record_id')['target'].mean()
global_mean = train_target['target'].mean()

train_target['id_te_train_only'] = train_target['record_id'].map(train_means)
test_target['id_te_train_only'] = test_target['record_id'].map(train_means).fillna(global_mean)

X_train_c = train_target[['id_te_train_only', 'noise_feature']]
X_test_c = test_target[['id_te_train_only', 'noise_feature']]
y_train_c = train_target['target']
y_test_c = test_target['target']

model_target_corrected = LogisticRegression(solver='liblinear', random_state=SEED)
model_target_corrected.fit(X_train_c, y_train_c)

pred_prob_c = model_target_corrected.predict_proba(X_test_c)[:, 1]
pred_cls_c = (pred_prob_c >= 0.5).astype(int)

aUC_target_corrected = roc_auc_score(y_test_c, pred_prob_c)
acc_target_corrected = accuracy_score(y_test_c, pred_cls_c)

print(f"Corrected ROC-AUC: {aUC_target_corrected:.4f}")
print(f"Corrected Accuracy: {acc_target_corrected:.4f}")

### Metric comparison and interpretation

In [ ]:
target_results = pd.DataFrame([
    {'pipeline': 'Flawed (full-data encoding)', 'roc_auc': aUC_target_flawed, 'accuracy': acc_target_flawed},
    {'pipeline': 'Corrected (train-only encoding)', 'roc_auc': aUC_target_corrected, 'accuracy': acc_target_corrected},
])

print(target_results.to_string(index=False))

ax = target_results.set_index('pipeline')[['roc_auc', 'accuracy']].plot(
    kind='bar', figsize=(8, 4), rot=0, ylim=(0.45, 1.05)
)
ax.set_title('Target Leakage: Flawed vs Corrected')
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()

The flawed pipeline appears excellent because each encoded value directly used label information from the same row.
After preventing leakage, performance collapses toward random, revealing there was no true signal in `record_id`.

## 2) Temporal leakage via random split on time-dependent data

### Setup
We generate a time series with lag features and a **late regime shift**. In real forecasting, training should contain only past data.

- A random split mixes early and late periods in both train and test.
- This gives the model access to future patterns during training.
- A chronological split avoids that leakage.

In [ ]:
n_time = 1400
series = np.zeros(n_time)
noise = rng.normal(0, 0.6, n_time)

for t in range(2, n_time):
    if t < 950:
        series[t] = 0.8 * series[t - 1] - 0.15 * series[t - 2] + noise[t]
    else:
        # Late regime shift with a level jump
        series[t] = 1.05 * series[t - 1] - 0.1 * series[t - 2] + 3.0 + noise[t]

df_time = pd.DataFrame({
    'time': np.arange(n_time),
    'target': series,
})
df_time['lag_1'] = df_time['target'].shift(1)
df_time['lag_2'] = df_time['target'].shift(2)
df_time = df_time.dropna().reset_index(drop=True)

print(df_time.head())
print(df_time.tail())

### Flawed pipeline (random split on temporal data)

This split shuffles samples, so training data includes information from later times than test samples.

In [ ]:
idx_train_t, idx_test_t = train_test_split(
    df_time.index, test_size=0.25, random_state=SEED, shuffle=True
)

train_time_flawed = df_time.loc[idx_train_t]
test_time_flawed = df_time.loc[idx_test_t]

X_train_tf = train_time_flawed[['lag_1', 'lag_2']]
X_test_tf = test_time_flawed[['lag_1', 'lag_2']]
y_train_tf = train_time_flawed['target']
y_test_tf = test_time_flawed['target']

model_time_flawed = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=3,
    random_state=SEED,
)
model_time_flawed.fit(X_train_tf, y_train_tf)

pred_tf = model_time_flawed.predict(X_test_tf)
rmse_time_flawed = np.sqrt(mean_squared_error(y_test_tf, pred_tf))
r2_time_flawed = r2_score(y_test_tf, pred_tf)

print(f"Random split train time range: {train_time_flawed['time'].min()} to {train_time_flawed['time'].max()}")
print(f"Random split test time range:  {test_time_flawed['time'].min()} to {test_time_flawed['time'].max()}")
print(f"Flawed RMSE: {rmse_time_flawed:.4f}")
print(f"Flawed R^2:  {r2_time_flawed:.4f}")

### Corrected pipeline (chronological split)

Train on the first 75% of time and test on the most recent 25%.

In [ ]:
split_point = int(len(df_time) * 0.75)

train_time_corrected = df_time.iloc[:split_point].copy()
test_time_corrected = df_time.iloc[split_point:].copy()

X_train_tc = train_time_corrected[['lag_1', 'lag_2']]
X_test_tc = test_time_corrected[['lag_1', 'lag_2']]
y_train_tc = train_time_corrected['target']
y_test_tc = test_time_corrected['target']

model_time_corrected = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=3,
    random_state=SEED,
)
model_time_corrected.fit(X_train_tc, y_train_tc)

pred_tc = model_time_corrected.predict(X_test_tc)
rmse_time_corrected = np.sqrt(mean_squared_error(y_test_tc, pred_tc))
r2_time_corrected = r2_score(y_test_tc, pred_tc)

print(f"Chronological train time range: {train_time_corrected['time'].min()} to {train_time_corrected['time'].max()}")
print(f"Chronological test time range:  {test_time_corrected['time'].min()} to {test_time_corrected['time'].max()}")
print(f"Corrected RMSE: {rmse_time_corrected:.4f}")
print(f"Corrected R^2:  {r2_time_corrected:.4f}")

### Metric comparison and interpretation

In [ ]:
temporal_results = pd.DataFrame([
    {'pipeline': 'Flawed (random split)', 'rmse': rmse_time_flawed, 'r2': r2_time_flawed},
    {'pipeline': 'Corrected (chronological split)', 'rmse': rmse_time_corrected, 'r2': r2_time_corrected},
])

print(temporal_results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
temporal_results.set_index('pipeline')[['rmse']].plot(kind='bar', rot=0, ax=axes[0], legend=False)
axes[0].set_title('RMSE (lower is better)')
axes[0].set_ylabel('RMSE')

temporal_results.set_index('pipeline')[['r2']].plot(kind='bar', rot=0, ax=axes[1], legend=False)
axes[1].set_title('R^2 (higher is better)')
axes[1].set_ylabel('R^2')

plt.tight_layout()
plt.show()

The random split overestimates performance because the model learns patterns from future periods.
Chronological validation is stricter and better reflects real-world forecasting where the future is unseen.

## 3) Group leakage via entity overlap

### Setup
We simulate repeated observations per entity (e.g., patient/customer/device), where target is mostly entity-specific.

- A row-wise random split lets the same entity appear in both train and test.
- If entity identity is encoded, models can memorize entity labels.
- Proper group-aware splitting prevents this overlap.

In [ ]:
n_entities = 500
rows_per_entity = 5

entity_ids = np.repeat(np.arange(n_entities), rows_per_entity)
entity_target = rng.binomial(1, 0.5, n_entities)
y_group = np.repeat(entity_target, rows_per_entity)

# Weak non-identity signal so corrected pipeline is not trivially impossible
weak_signal = 0.25 * y_group + rng.normal(0, 1.0, len(entity_ids))

df_group = pd.DataFrame({
    'entity_id': entity_ids.astype(str),
    'weak_signal': weak_signal,
    'target': y_group,
})

print(df_group.head())
print(f"Rows: {len(df_group)}, unique entities: {df_group['entity_id'].nunique()}")

### Flawed pipeline (row-wise random split)

The same entities leak across train and test.

In [ ]:
X_group = df_group[['entity_id', 'weak_signal']]
y_group_all = df_group['target']

X_train_gf, X_test_gf, y_train_gf, y_test_gf = train_test_split(
    X_group, y_group_all, test_size=0.3, random_state=SEED, stratify=y_group_all
)

pipeline_group_flawed = Pipeline(steps=[
    ('prep', ColumnTransformer(transformers=[
        ('entity', OneHotEncoder(handle_unknown='ignore'), ['entity_id']),
        ('num', StandardScaler(), ['weak_signal']),
    ])),
    ('model', LogisticRegression(max_iter=1000, solver='liblinear', random_state=SEED)),
])

pipeline_group_flawed.fit(X_train_gf, y_train_gf)

pred_prob_gf = pipeline_group_flawed.predict_proba(X_test_gf)[:, 1]
pred_cls_gf = (pred_prob_gf >= 0.5).astype(int)

auc_group_flawed = roc_auc_score(y_test_gf, pred_prob_gf)
acc_group_flawed = accuracy_score(y_test_gf, pred_cls_gf)

overlap_flawed = len(set(X_train_gf['entity_id']).intersection(set(X_test_gf['entity_id'])))

print(f"Entity overlap count (flawed split): {overlap_flawed}")
print(f"Flawed ROC-AUC: {auc_group_flawed:.4f}")
print(f"Flawed Accuracy: {acc_group_flawed:.4f}")

### Corrected pipeline (group-aware split)

`GroupShuffleSplit` ensures entity IDs do not overlap between train and test.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
train_idx_g, test_idx_g = next(gss.split(X_group, y_group_all, groups=df_group['entity_id']))

X_train_gc = X_group.iloc[train_idx_g].copy()
X_test_gc = X_group.iloc[test_idx_g].copy()
y_train_gc = y_group_all.iloc[train_idx_g].copy()
y_test_gc = y_group_all.iloc[test_idx_g].copy()

pipeline_group_corrected = Pipeline(steps=[
    ('prep', ColumnTransformer(transformers=[
        ('entity', OneHotEncoder(handle_unknown='ignore'), ['entity_id']),
        ('num', StandardScaler(), ['weak_signal']),
    ])),
    ('model', LogisticRegression(max_iter=1000, solver='liblinear', random_state=SEED)),
])

pipeline_group_corrected.fit(X_train_gc, y_train_gc)

pred_prob_gc = pipeline_group_corrected.predict_proba(X_test_gc)[:, 1]
pred_cls_gc = (pred_prob_gc >= 0.5).astype(int)

auc_group_corrected = roc_auc_score(y_test_gc, pred_prob_gc)
acc_group_corrected = accuracy_score(y_test_gc, pred_cls_gc)

overlap_corrected = len(set(X_train_gc['entity_id']).intersection(set(X_test_gc['entity_id'])))

print(f"Entity overlap count (corrected split): {overlap_corrected}")
print(f"Corrected ROC-AUC: {auc_group_corrected:.4f}")
print(f"Corrected Accuracy: {acc_group_corrected:.4f}")

### Metric comparison and interpretation

In [ ]:
group_results = pd.DataFrame([
    {'pipeline': 'Flawed (row split)', 'roc_auc': auc_group_flawed, 'accuracy': acc_group_flawed, 'entity_overlap': overlap_flawed},
    {'pipeline': 'Corrected (group split)', 'roc_auc': auc_group_corrected, 'accuracy': acc_group_corrected, 'entity_overlap': overlap_corrected},
])

print(group_results.to_string(index=False))

ax = group_results.set_index('pipeline')[['roc_auc', 'accuracy']].plot(
    kind='bar', figsize=(8, 4), rot=0, ylim=(0.45, 1.05)
)
ax.set_title('Group Leakage: Flawed vs Corrected')
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()

With row-wise splitting, shared entities let the model memorize identity-related patterns and inflate test scores.
With group-aware splitting, entity overlap is zero, and metrics become a more realistic estimate for unseen entities.

## Key takeaways

- **Target leakage:** compute target-dependent statistics strictly inside training folds/splits.
- **Temporal leakage:** use time-aware validation (chronological split / rolling windows).
- **Group leakage:** split by entity (`GroupKFold`, `GroupShuffleSplit`) when observations are correlated within groups.

Leakage almost always looks like “great validation, poor production.” Defensive split design is your first line of protection.